In [1]:
"""
Figure 1 — OWAGDP aggregate across the orness grid.

Run as a follow-on CELL after 02_aggregate.py in the same notebook: aggregates.csv
is already on disk. Nothing is recomputed -- the lines ARE the owa_0.1 ... owa_0.9
columns, so the figure cannot disagree with Table II.

LAYOUT
  WIDE = False  single column (3.5 in), panel (a) only.        <- default
                Recommended now that Fig. 2 is being cut: the spread histogram's
                content (median 0.16, five cells under 0.05, max 2.24 at Ireland)
                is already stated in the text, so it is the cheapest thing to
                drop. Use \\begin{figure}.
  WIDE = True   full width (7.16 in), panels (a) and (b).
                Use \\begin{figure*}.

WRITES  fig1.pdf, fig1.png
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WIDE = False

# One steep case, one moderate, one flat -- so the panel shows the range of
# behaviour rather than only Ireland.
SERIES = [("IRL", 2026), ("SWE", 2027), ("ITA", 2027)]
GRID = [round(0.1 * i, 1) for i in range(1, 10)]

# Validated categorical order: worst adjacent CVD dE 24.7, normal-vision 33.6.
# Marker and dash also differ, so the panel survives grayscale printing.
COLORS = ["#2a78d6", "#eb6834", "#4a3aa7", "#1baf7a"]
MARKERS = ["o", "s", "^", "D"]
DASHES = [(None, None), (4, 1.6), (1.2, 1.2), (5, 1.5, 1.2, 1.5)]
INK, INK_2, GRIDC = "#0b0b0b", "#52514e", "#c9c8c2"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Nimbus Roman No9 L", "Times New Roman", "DejaVu Serif"],
    "font.size": 7.2, "axes.labelsize": 7.4, "axes.linewidth": 0.5,
    "xtick.labelsize": 7.0, "ytick.labelsize": 7.0,
    "pdf.fonttype": 42, "ps.fonttype": 42,     # PDF eXpress rejects Type 3
})

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQ = {"iso", "year", "spread", *[f"owa_{a}" for a in GRID]}


def load():
    for f in ("aggregates.csv",) + tuple(sorted(os.listdir("."))):
        if not f.lower().endswith(".csv"):
            continue
        try:
            if REQ.issubset(pd.read_csv(f, nrows=1).columns):
                return pd.read_csv(f)
        except Exception:
            pass
    if IN_COLAB:
        print("Upload aggregates.csv")
        files.upload()
        return load()
    raise FileNotFoundError("aggregates.csv not found -- run 02_aggregate.py first")


d = load()

if WIDE:
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.16, 2.55))
else:
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    bx = None

# ------------------------------------------------------------- the grid ----
ax.axhline(0, color=INK_2, lw=0.5, zorder=1)
ax.axvline(0.5, color=GRIDC, lw=0.5, ls=(0, (1.5, 1.5)), zorder=1)
ends = []
for (iso, yr), c, mk, dash in zip(SERIES, COLORS, MARKERS, DASHES):
    r = d[(d.iso == iso) & (d.year == yr)]
    if r.empty:
        print(f"  warning: {iso} {yr} not in the file, skipped")
        continue
    y = [r.iloc[0][f"owa_{a}"] for a in GRID]
    ax.plot(GRID, y, color=c, lw=1.1, marker=mk, ms=2.8, mec="white", mew=0.4,
            dashes=dash, zorder=3)
    ends.append([y[-1], y[-1], f"{iso} {yr}", c])

# direct labels, pushed apart when two series finish close together, so identity
# never rests on colour alone
ends.sort(key=lambda e: e[1])
lo, hi = ax.get_ylim()
gap = 0.085 * (hi - lo)
for i in range(1, len(ends)):
    if ends[i][1] - ends[i - 1][1] < gap:
        ends[i][1] = ends[i - 1][1] + gap
for y_true, y_lab, txt, c in ends:
    if abs(y_lab - y_true) > 1e-9:
        ax.plot([GRID[-1], GRID[-1] + 0.028], [y_true, y_lab], color=c, lw=0.4,
                clip_on=False, zorder=2)
    ax.text(GRID[-1] + 0.034, y_lab, txt, va="center", ha="left", fontsize=6.6,
            color=c, clip_on=False)

ax.set_xlabel(r"Orness level $\alpha$")
ax.set_ylabel("OWAGDP aggregate (per cent growth)")
ax.set_xticks([0.1, 0.3, 0.5, 0.7, 0.9] if not WIDE else GRID)
ax.set_xlim(0.06, 0.94)
ax.grid(axis="y", color=GRIDC, lw=0.35, alpha=0.7, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
if WIDE:
    ax.set_title("(a) Aggregate across the orness grid", fontsize=7.4, pad=4)

# --------------------------------------------------- optional histogram ----
if bx is not None:
    sp = d.spread.dropna().values
    med = np.median(sp)
    bx.hist(sp, bins=np.arange(0, sp.max() + 0.1, 0.1), color=COLORS[0],
            alpha=0.85, edgecolor="white", linewidth=0.4, zorder=2)
    bx.axvline(med, color=INK, lw=0.9, ls=(0, (3, 1.5)), zorder=3)
    bx.annotate(f"median {med:.2f}", xy=(med, bx.get_ylim()[1]), xytext=(4, -4),
                textcoords="offset points", va="top", ha="left", fontsize=6.8,
                color=INK)
    i = int(np.argmax(sp))
    bx.annotate(f"{d.iloc[i].iso} {d.iloc[i].year}", xy=(sp[i], 1),
                xytext=(-6, 26), textcoords="offset points", ha="right",
                fontsize=6.8, color=INK,
                arrowprops=dict(arrowstyle="-", lw=0.5, color=INK_2))
    bx.set_xlabel(r"Attitudinal spread, $\alpha=0.8$ minus $\alpha=0.2$ (pp)")
    bx.set_ylabel(f"Number of country-years (n={len(sp)})")
    bx.grid(axis="y", color=GRIDC, lw=0.35, alpha=0.7, zorder=0)
    bx.set_axisbelow(True)
    for s in ("top", "right"):
        bx.spines[s].set_visible(False)
    bx.set_title(f"(b) Spread distribution ({d.iso.nunique()} economies "
                 r"$\times$ 2 years)", fontsize=7.4, pad=4)

plt.tight_layout(pad=0.3)
if WIDE:
    fig.subplots_adjust(left=0.085, right=0.90, wspace=0.42)
else:
    fig.subplots_adjust(right=0.74)          # room for the direct labels
for ext in ("pdf", "png"):
    fig.savefig(f"fig1.{ext}", dpi=400, bbox_inches="tight", pad_inches=0.02)
plt.show()

w, h = fig.get_size_inches()
print(f"wrote fig1.pdf ({w:.2f} x {h:.2f} in) -- use "
      + ("\\begin{figure*}" if WIDE else "\\begin{figure}"))
for iso, yr in SERIES:
    r = d[(d.iso == iso) & (d.year == yr)]
    if not r.empty:
        r = r.iloc[0]
        print(f"  {iso} {yr}: {r['owa_0.1']:+.2f} at a=0.1 -> "
              f"{r['owa_0.9']:+.2f} at a=0.9  (spread {r.spread:.3f})")

if IN_COLAB:
    files.download("fig1.pdf")

Upload aggregates.csv


Saving aggregates.csv to aggregates.csv
wrote fig1.pdf (3.50 x 2.50 in) -- use \begin{figure}
  IRL 2026: -1.10 at a=0.1 -> +1.89 at a=0.9  (spread 2.242)
  SWE 2027: +1.96 at a=0.1 -> +2.47 at a=0.9  (spread 0.383)
  ITA 2027: +0.51 at a=0.1 -> +0.59 at a=0.9  (spread 0.058)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>